In [37]:
import ee
import rasterio
from os import path

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='donnie-creek-wildfire')


In [38]:
# Import the MODIS land cover collection.
lc = ee.ImageCollection('MODIS/006/MCD12Q1')

# Import the MODIS land surface temperature collection.
lst = ee.ImageCollection('MODIS/006/MOD11A1')

# Import the USGS ground elevation image.
elv = ee.Image('USGS/SRTMGL1_003')

In [39]:
# Initial date of interest (inclusive)
i_date = '2017-01-01'

# Final date of interest (exlusive)
f_date = '2020-01-01'

# Selection of appropriae bands and dates for LST
lst = lst.select('LST_Day_1km', 'QC_Day').filterDate(i_date, f_date)

In [40]:
# Define the urban location of interest as a point near Lyon, France
u_lon = 4.8148
u_lat = 45.7758
u_poi = ee.Geometry.Point(u_lon, u_lat)

# Define the rural location of interest as a point away from the city
r_lon = 5.175964
r_lat = 45.574064
r_poi = ee.Geometry.Point(r_lon, r_lat)

In [42]:
scale = 1000 # scale in meters

# Print the elevation near Lyon, France
elev_urban_point = elv.sample(u_poi, scale).first().get('elevation').getInfo()
print(f"Elevation near Lyon, France: {elev_urban_point} m")

# Calculate and print the mean value of the LST collection at the point
lst_urban_point = lst.mean().sample(u_poi, scale).first().get('LST_Day_1km').getInfo()
print('Average daytime LST at urban point:', round(lst_urban_point * 0.02 - 273.15, 2), '°C') 
# LST values are corrected by a factor of 0.02 and in Kelvin, convert to °C

# Print the land cover type at the point
lc_urban_point = lc.first().sample(u_poi, scale).first().get('LC_Type1').getInfo()
print('Land cover type at urban point is:', lc_urban_point)

Elevation near Lyon, France: 196 m
Average daytime LST at urban point: 23.12 °C
Land cover type at urban point is: 13


In [44]:
# Get the data for the pixel intersecting the point in urban area
lst_u_poi = lst.getRegion(u_poi, scale).getInfo()

# Get the data for the pixel intersecting the point in rural area
lst_r_poi = lst.getRegion(r_poi, scale).getInfo()

# Preview the result
lst_u_poi[:5]

[['id', 'longitude', 'latitude', 'time', 'LST_Day_1km', 'QC_Day'],
 ['2017_01_01', 4.810478346460038, 45.77365530231022, 1483228800000, None, 2],
 ['2017_01_02', 4.810478346460038, 45.77365530231022, 1483315200000, None, 2],
 ['2017_01_03', 4.810478346460038, 45.77365530231022, 1483401600000, None, 2],
 ['2017_01_04',
  4.810478346460038,
  45.77365530231022,
  1483488000000,
  13808,
  17]]